(placeholder title)

(placeholder plan)

(placeholder setup-heading)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression, RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer

import cleverly

pd.set_option("display.width", 130)
print("cleverly", cleverly.__version__)

(placeholder data-heading)

In [ ]:
from cleverly.datasets import navigation_data

# With your own data, load your data frame here, such as frame = pd.read_csv(...).
# Real data have no `truth`, so drop every line that uses it.
frame, truth = navigation_data(n=1_000, seed=52)
# The characteristics to adjust for.
covariates = ["discharge_risk", "prior_utilization", "medication_burden", "age"]
print("rows and columns:", frame.shape)
print(list(frame.columns))
print(frame.head().round(3))

(placeholder data-reading)

(placeholder association-heading)

In [ ]:
by_arm = frame.groupby("transition_navigation")["transition_score"].mean()
print(f"unadjusted difference in mean score: {by_arm[1.0] - by_arm[0.0]:.3f}")
print(f"population ATE:                      {truth['ate']:.3f}")

(placeholder association-reading)

(placeholder protocol-heading)

In [ ]:
from cleverly.datasets import navigation_protocol

# This page's program comes with its study description already filled in; the
# point-treatment example explains each field.
protocol = navigation_protocol()
print("\n".join(protocol.summary_lines()))

In [ ]:
from cleverly import ATE, CausalStudy, PointTreatment

# For your own data, change the outcome and treatment names and the `covariates` list.
study = CausalStudy(
    frame,
    design=PointTreatment(
        outcome="transition_score",
        treatment="transition_navigation",
        adjustment=tuple(covariates),
    ),
    protocol=protocol,
)
effect = study.identify(ATE(reference=0))
print(effect.summary())

(placeholder identify-reading)

(placeholder plan-fits-heading)

In [ ]:
from cleverly import CrossFitting, Runtime, Targeting, TMLEMethod
from cleverly.learners import default_library

print("default candidates:", [name for name, _ in default_library("classification")])
# The method of the default fit: no ModelSpec, so both models use the default ensemble.
# Five outer folds instead of the default ten keep this page's run short.
default_method = TMLEMethod(
    cross_fitting=CrossFitting(n_folds=5),
    targeting=Targeting(q_bounds=(0.0, 1.0)),
    runtime=Runtime(random_state=52, n_jobs=1),
)
planned = effect.planned_fits(method=default_method)
print("model fits:    ", planned.total_fits)
print("candidate fits:", planned.total_candidate_fits)
print(planned.summary())

(placeholder plan-fits-reading)

(placeholder declared-fit-heading)

In [ ]:
from cleverly import ModelSpec, SuperLearner


def shallow_booster(kind):
    """Gradient boosting with two-level trees and a penalty, so it cannot memorize rows."""
    return kind(max_depth=2, learning_rate=0.05, max_iter=200, l2_regularization=1.0)


# For your own data, list the candidates you would consider, from simple to flexible.
outcome_ensemble = SuperLearner(
    library=[
        ("linear", LinearRegression()),
        ("splines", make_pipeline(SplineTransformer(n_knots=5), RidgeCV())),
        ("boosting", shallow_booster(HistGradientBoostingRegressor)),
    ],
    n_folds=3,
    clip=(0.0, 1.0),
    random_state=52,
)
treatment_ensemble = SuperLearner(
    library=[
        ("logistic", LogisticRegression(max_iter=1000)),
        ("splines", make_pipeline(SplineTransformer(n_knots=5), LogisticRegression(max_iter=2000))),
        ("boosting", shallow_booster(HistGradientBoostingClassifier)),
    ],
    n_folds=3,
    random_state=52,
)
declared_method = TMLEMethod(
    models=ModelSpec(outcome_learner=outcome_ensemble, treatment_learner=treatment_ensemble),
    cross_fitting=CrossFitting(n_folds=5),
    targeting=Targeting(q_bounds=(0.0, 1.0)),
    runtime=Runtime(random_state=52, n_jobs=1),
)
declared = effect.estimate(method=declared_method)
estimate = declared["ate"]
print(
    f"declared ensembles: ATE {estimate.psi:.4f}, standard error {estimate.std_error:.4f}, "
    f"95% interval {estimate.ci[0]:.4f} to {estimate.ci[1]:.4f}"
)
print(f"population ATE:     {truth['ate']:.4f}")
print("candidate fits:", effect.planned_fits(method=declared_method).total_candidate_fits)

(placeholder declared-fit-reading)

(placeholder weights-heading)

In [ ]:
from cleverly import plotting


def weight_table(result):
    """Each model's candidates with their cross-validated risk and ensemble weight."""
    rows = []
    for model in result.diagnostics.nuisance_models().models:
        label = {"propensity": "treatment model", "outcome": "outcome model"}[model.name]
        for name, weight in model.learner_weights.items():
            rows.append(
                {
                    "model": label,
                    "candidate": name,
                    "cross-validated risk": model.learner_risks[name],
                    "weight": weight,
                }
            )
    return pd.DataFrame(rows)


weights = weight_table(declared)
print(weights.round(4).to_string(index=False))
learners_fig = plotting.learners(declared)

(placeholder weights-reading)

(placeholder default-fit-heading)

In [ ]:
# The default ensemble, fitted once on every row for each model, as each outer fold of
# the default fit would: five inner folds, and predicted scores kept inside 0 to 1.
offer = frame["transition_navigation"].to_numpy()
score = frame["transition_score"].to_numpy()
default_treatment = SuperLearner(task="classification", n_folds=5, random_state=52)
default_treatment.fit(frame[covariates].to_numpy(), offer)
default_outcome = SuperLearner(task="regression", n_folds=5, clip=(0.0, 1.0), random_state=52)
default_outcome.fit(frame[["transition_navigation", *covariates]].to_numpy(), score)
rows = []
for label, ensemble in (("treatment model", default_treatment), ("outcome model", default_outcome)):
    found = ensemble.diagnostics_
    for name, risk, weight in zip(found.names, found.cv_risk, found.weights, strict=True):
        rows.append(
            {"model": label, "candidate": name, "cross-validated risk": risk, "weight": weight}
        )
print(pd.DataFrame(rows).round(4).to_string(index=False))

(placeholder default-fit-reading)

(placeholder screening-heading)

In [ ]:
from dataclasses import replace

from cleverly.learners import screen_by_outcome
from cleverly.learners.screeners import default_min_retain

# Twenty extra recorded columns of pure noise: they affect neither the offer nor the score,
# so the true effect is unchanged.
rng = np.random.default_rng(52)
noise = pd.DataFrame(
    rng.normal(size=(len(frame), 20)), columns=[f"record_{i:02d}" for i in range(20)]
)
wide = pd.concat([frame, noise], axis=1)
wide_covariates = [*covariates, *noise.columns]
wide_effect = CausalStudy(
    wide,
    design=PointTreatment(
        outcome="transition_score",
        treatment="transition_navigation",
        adjustment=tuple(wide_covariates),
    ),
    protocol=protocol,
).identify(ATE(reference=0))
# One shallow booster per model keeps 24 columns quick; boosting can use curved effects.
plain = replace(
    declared_method,
    models=ModelSpec(
        outcome_learner=shallow_booster(HistGradientBoostingRegressor),
        treatment_learner=shallow_booster(HistGradientBoostingClassifier),
    ),
)
# For your own data, min_retain is the fewest columns the screen keeps.
screened_models = replace(plain.models, screen_treatment=True, min_retain=4)
unscreened = wide_effect.estimate(method=plain)
screened = wide_effect.estimate(method=replace(plain, models=screened_models))
for label, fitted in (("all 24 columns", unscreened), ("screened", screened)):
    row = fitted["ate"]
    print(f"{label:15s} ATE {row.psi:.4f}, standard error {row.std_error:.4f}")
# The same screen on every row, to see which columns it keeps.
x = wide[wide_covariates].to_numpy()
kept = screen_by_outcome(
    x,
    wide["transition_navigation"].to_numpy(),
    wide["transition_score"].to_numpy(),
    min_retain=4,
)
arms = wide["transition_navigation"].value_counts().tolist()
floor = default_min_retain(len(wide_covariates), len(wide), arms)
print("columns the default floor keeps at the least:", floor, "of", len(wide_covariates))
print(
    "columns the screen keeps on all rows:", [str(name) for name in np.array(wide_covariates)[kept]]
)

(placeholder screening-reading)

(placeholder assessment-heading)

In [ ]:
assessment = declared.assess()
support = assessment.report("support")
nuisance = assessment.report("nuisance_models")
scores = assessment.report("score_equations")
(propensity,) = (model for model in nuisance.models if model.name == "propensity")
(outcome_model,) = (model for model in nuisance.models if model.name == "outcome")
attention = tuple(item.name for item in assessment.attention)
print("checks that need attention:", attention or "none")
print()
print("overlap, from the fitted treatment model:")
smallest = support.propensity_quantiles["overall"][0.0]
print(f"  smallest predicted offer probability: {smallest:.4f}")
print(f"  share of patients at the weight cap:  {support.truncated['fraction']:.4f}")
for arm, size in support.effective_sample_size.items():
    share = f"{size['effective']:.0f} of {size['n']:.0f} ({size['ratio']:.2f})"
    print(f"  effective sample size, {arm:7s}: {share}")
print()
slope = propensity.metrics["calibration_slope"]
slope_se = propensity.metrics["calibration_slope_se"]
print(f"treatment model calibration slope: {slope:.2f} (standard error {slope_se:.2f})")
print(f"outcome model R^2: {outcome_model.metrics['r2']:.2f}")
print(nuisance.summary().reading)
print(scores.summary().reading)
overlap_fig = plotting.overlap(declared, labels={1.0: "offered", 0.0: "not offered"})

(placeholder assessment-reading)

(placeholder sensitivity-heading)

In [ ]:
robustness = declared.sensitivity.robustness_value()
print(f"robustness value:                  {robustness['rv']:.3f}")
print(f"robustness value at the 95% limit: {robustness['rva']:.3f}")

(placeholder sensitivity-reading)

(placeholder interpret)

(placeholder go-deeper)

(placeholder refutation-heading)

In [ ]:
from cleverly.validation import refute

checks = refute(
    declared,
    tests=("placebo", "random_common_cause", "subset"),
    n_replicates=5,
)
columns = ["test", "original", "refuted_mean", "refuted_sd", "expectation", "verdict"]
print(checks.to_frame()[columns].round(4).to_string(index=False))

(placeholder refutation-reading)

(placeholder method-evidence)

(placeholder discrete-heading)

In [ ]:
from sklearn.base import clone

# meta_learner="discrete" keeps only the candidate with the smallest cross-validated risk.
discrete_models = ModelSpec(
    outcome_learner=clone(outcome_ensemble).set_params(meta_learner="discrete"),
    treatment_learner=clone(treatment_ensemble).set_params(meta_learner="discrete"),
)
discrete = effect.estimate(method=replace(declared_method, models=discrete_models))
row = discrete["ate"]
print(f"one candidate per model: ATE {row.psi:.4f}, standard error {row.std_error:.4f}")
print(weight_table(discrete).round(4).to_string(index=False))

(placeholder discrete-reading)

(placeholder where-next)